In [ ]:
import re
import os
import json
from tqdm import tqdm 
from datasets import load_dataset


# 读取数据
def read_datas(path):
    with open(path, 'r', encoding='utf8') as f:
        for line in f:
            yield line

# 解析 json 字符串
def extract_json_from_string(text):
    """
    使用正则表达式从字符串中宽松地提取 JSON 字符串。
    这个正则表达式会查找以 '{' 开头，以 '}' 结尾，并且中间可以包含任何字符（包括换行符）的模式。
    """
    # 宽松的 JSON 匹配，考虑多行
    # re.DOTALL 使 '.' 匹配包括换行符在内的所有字符
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
    except: 
        return None
    if match:
        json_str = match.group(0)
        try:
            # 尝试解析提取到的字符串，确保它是有效的 JSON
            json.loads(json_str)
            return json_str
        except json.JSONDecodeError:
            # 如果不是有效的 JSON，则返回 None
            return None
    return None

def process_jsonl_file(path):
    """
    处理 JSONL 文件，提取 JSON 数据并返回一个列表。
    同时记录提取成功和失败的数量。
    """
    extracted_data = []
    sources = []
    prompts = []
    total_lines = 0
    successful_extractions = 0
    failed_extractions = 0

    for line in tqdm(read_datas(path)):
        cur_ = json.loads(line)
        line = cur_['output']

        extracted_json_str = extract_json_from_string(line)
        if extracted_json_str:
            try:
                # extracted_data.append(json.loads(extracted_json_str))
                extracted_data.append({'source': cur_['source'], 'prompt': cur_['prompt'], 'score': json.loads(extracted_json_str)})
                successful_extractions += 1
                # sources.append(cur_['source'])
                # prompts.append(cur_['prompt'])
                
            except json.JSONDecodeError as e:
                print(f"警告: 第 {line_num} 行提取的字符串不是有效的JSON，忽略。错误: {e}")
                failed_extractions += 1
        else:
            failed_extractions += 1

    return extracted_data, sources, prompts, total_lines, successful_extractions, failed_extractions

In [ ]:
# 格式筛选，生成结果中存在json 无法正常解析的直接删除
# extracted_data_1 = process_jsonl_file('/code/zhaoxudong03/data_pipelines/training_data_zxd/totals_datas_cls.jsonl')
# extracted_data_2 = process_jsonl_file('/code/zhaoxudong03/data_pipelines/training_data_zxd/totals_datas_cls_2.jsonl')

extracted_data_1 = process_jsonl_file('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/total_202w_only_156w_cat_nvidia-2_stages_204w_total_360w_02_embeded_dep_extract_for_30b_a3b_judge.jsonl_cls_before.jsonl')


In [ ]:
# 过滤掉数据中，划分维度： 指令清晰度（5分以上可用）， 认知复杂度（区分是否困难，第一次不作为标准，easydistill 中筛选的是认知难度为 4分的数据)， 风险类别筛选 安全标签
def filter(datas):
    results = []
    no_clarity = []
    no_safe = []
    error = 0 
    for elem in datas:
        # if not ('Clarity' in elem['score'] and 'Safety_Risk' in elem['score'] and 'Cognitive_Complexity' in elem['score'] and 'Domain' in elem['score']):
        #     print(elem)
        #     break
        # 部分数据，格式存在问题，比例很少，采用直接删除方式处理
        try:
            if elem['score']['Clarity']['Score'] >= 0 and elem['score']['Safety_Risk']['Category'].lower() == 'safe':
                results.append(elem)
            # elif elem['score']['Clarity']['Score'] < 5:
            #     no_clarity.append(elem)
            elif elem['score']['Safety_Risk']['Category'].lower() != 'safe':
                no_safe.append(elem)
        except Exception as e:
            error += 1
            continue
    return results, no_clarity, no_safe, error

In [ ]:
filter_res_1, no_clarity_1, no_safe_1, error = filter(extracted_data_1[0])


In [ ]:
filter_res_2, no_clarity_2, no_safe_2, error_2 = filter(extracted_data_2[0])
print(len(filter_res_1), len(no_clarity_1), len(no_safe_1), error)
# print(len(filter_res_2), len(no_clarity_2), len(no_safe_2), error_2)

In [ ]:
# 初步过滤剩余 filter_res_1, filter_res_2， 这些是可用切安全的数据
# len(filter_res_1) + len(filter_res_2)

# 第二步过滤
# 1、先按照 数据类型分类，保存文件， 发现分类标签多处很多，模型自己总结的分类，但是数据比例较低，86w中有 1k， 因此直接归类在 other中
def cls_datas_for_save(datas):
    data_dics = {}
    for elem in datas:
        try:
            cur_cls = elem['score']['Domain']['Primary_Domain']
            if cur_cls in data_dics:
                data_dics[cur_cls].append(elem)
            else:
                data_dics[cur_cls] = [elem]
        except Exception as e:
            continue

    return data_dics

In [ ]:
data_type_1 = cls_datas_for_save(filter_res_1)
data_type_2 = cls_datas_for_save(filter_res_2)


In [ ]:
count_ = 0
for type_ in ['Code', 'Math', 'Reasoning_&_Logic', 'Data_Analysis', 'Creative_Writing', 'Instruction_Following_&_Text_Processing', 'Role_Playing_&_Chat', 'Humanities_Arts_&_Social_Sciences', 'Natural_&_Applied_Sciences', 'Business_&_Finance', 'Daily_Life_&_Health', 'General_Knowledge_QA', 'Other']:
    print(type_, len(data_type_1[type_]))
    count_ += len(data_type_1[type_])

In [ ]:
count_ = 0
for type_ in ['Code', 'Math', 'Reasoning_&_Logic', 'Data_Analysis', 'Creative_Writing', 'Instruction_Following_&_Text_Processing', 'Role_Playing_&_Chat', 'Humanities_Arts_&_Social_Sciences', 'Natural_&_Applied_Sciences', 'Business_&_Finance', 'Daily_Life_&_Health', 'General_Knowledge_QA', 'Other']:
    print(type_, len(data_type_1[type_]))
    count_ += len(data_type_1[type_])

In [ ]:
count_2 = 0
for type_ in ['Code', 'Math', 'Reasoning_&_Logic', 'Data_Analysis', 'Creative_Writing', 'Instruction_Following_&_Text_Processing', 'Role_Playing_&_Chat', 'Humanities_Arts_&_Social_Sciences', 'Natural_&_Applied_Sciences', 'Business_&_Finance', 'Daily_Life_&_Health', 'General_Knowledge_QA', 'Other']:
    print(type_, len(data_type_2[type_]))
    count_2 += len(data_type_2[type_])

In [ ]:
print(count_, len(filter_res_1) - count_ )
print(count_2, len(filter_res_2) - count_2 )

In [ ]:
# 合并所有数据
all_userful_datas = filter_res_1 + filter_res_2
len(all_userful_datas)
data_type_all = cls_datas_for_save(all_userful_datas)


In [ ]:
count_all = 0
for type_ in ['Code', 'Math', 'Reasoning_&_Logic', 'Data_Analysis', 'Creative_Writing', 'Instruction_Following_&_Text_Processing', 'Role_Playing_&_Chat', 'Humanities_Arts_&_Social_Sciences', 'Natural_&_Applied_Sciences', 'Business_&_Finance', 'Daily_Life_&_Health', 'General_Knowledge_QA', 'Other']:
    print(type_, len(data_type_all[type_]))
    count_all += len(data_type_all[type_])

In [ ]:
## step1 文件夹下是所有可用的数据
save_type = ['Code', 'Math', 'Reasoning_&_Logic', 'Data_Analysis', 'Creative_Writing', 'Instruction_Following_&_Text_Processing', 'Role_Playing_&_Chat', 'Humanities_Arts_&_Social_Sciences', 'Natural_&_Applied_Sciences', 'Business_&_Finance', 'Daily_Life_&_Health', 'General_Knowledge_QA']

import os
def save_datas(path, values):
    with open(path, 'w', encoding='utf8') as f_save:
        for value in values:
            f_save.write(json.dumps(value, ensure_ascii=False)+'\n')
    print('数据保存完成:::', path)
        

# 分类保存数据，将数据中不再定义标签内的分类，规划到Other中
def save_datas_by_cls(datas):
    others = []
    for k, v in datas.items():
        if k in save_type:
            save_path = os.path.join('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/step1', '{}.jsonl'.format(k))
            save_datas(save_path, v)
        else:
            others.extend(v)
    # 保存 Others
    save_datas('/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/step1/Others.jsonl', others)
# 开始分类别保存，
# save_datas = data_type_all

In [ ]:
data_type_1.keys()


In [ ]:
# save_datas_by_cls(data_type_all)
save_datas_by_cls(data_type_1)

In [ ]:
## step2 是根据规则筛选的数据，先采用模型打印，然后后续在批量打印
data_type_all['Code'][0]
# data_types = save_type + ['Other']

def process_datas(k, values, v_clarity_down, v_cg_down, v_clarity_up=10, v_cg_up=10):
    # 筛选条件
    res = []
    for elem in values:
        # print(elem['score']['Clarity']['Score'])
        if elem['score']['Clarity']['Score'] >= v_clarity_down and elem['score']['Clarity']['Score'] <= v_clarity_up and elem['score']['Cognitive_Complexity']['Score'] >= v_cg_down and elem['score']['Cognitive_Complexity']['Score'] <= v_cg_up:
            res.append(elem)

    return res
        
def deal_datas_step2(datas, v_clarity_down=8, v_clarity_up=10, v_cg_down=5, v_cg_up=10):
    total_results = []
    others = []
    for k, v in datas.items():
        if k in save_type:
            # 处理数据，传入阈值
            res = process_datas(k, v, v_clarity_down, v_cg_down, v_clarity_up, v_cg_up)
            total_results.append({k: res})
        else:
            others.extend(v)
    res = process_datas('Other', others, v_clarity_down, v_cg_down)
    total_results.append({'Other': res})
    return total_results

def step2_save_filter_datas_gai(data_type_all, v_clarity_down, v_cg_down, v_clarity_up=10, v_cg_up=10):
    str_name = 'clarity_over_{}_to_{}_and_cg_over_{}_to_{}'.format(v_clarity_down, v_clarity_up, v_cg_down, v_cg_up)
    res = deal_datas_step2(data_type_all, v_clarity_down=v_clarity_down, v_cg_down=v_cg_down, v_clarity_up=v_clarity_up, v_cg_up=v_cg_up)
    
    path = '/code/zhaoxudong03/data_pipelines/training_data_zxd_v2/data/step2'
    save_path = path + '/' + str_name
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    df_dics = {}
    for elem in res:
        for k, v in elem.items():
            df_dics[k] = len(v)
            # print(k, len(v))
            save_path_ = os.path.join(save_path, "{}.jsonl".format(k))
            save_datas(save_path_, v)

    return df_dics


In [ ]:
# clarity_over_8_and_cg_over_8 = deal_datas_step2(data_type_all, v_clarity_down=8, v_cg_down=8)

# def step2_save_filter_datas(rule):
#     path = '/code/zhaoxudong03/data_pipelines/training_data_zxd/step2'
#     save_path = path + '/' + rule
#     if not os.path.exists(save_path):
#         os.makedirs(save_path)
        
#     for elem in eval(rule):
#         for k, v in elem.items():
#             # print(k, len(v))
#             save_path_ = os.path.join(save_path, "{}.jsonl".format(k))
#             save_datas(save_path_, v)

# step2_save_filter_datas('clarity_over_8_and_cg_over_8')

In [ ]:
df_res = step2_save_filter_datas_gai(data_type_1, 8, 8)


In [ ]:
df_res = step2_save_filter_datas_gai(8, 8)


In [ ]:
df_res = {'General_Knowledge_QA': 6475,
 'Instruction_Following_&_Text_Processing': 3139,
 'Reasoning_&_Logic': 129003,
 'Math': 248583,
 'Daily_Life_&_Health': 1302,
 'Creative_Writing': 8097,
 'Natural_&_Applied_Sciences': 116346,
 'Humanities_Arts_&_Social_Sciences': 92534,
 'Role_Playing_&_Chat': 213,
 'Code': 119916,
 'Data_Analysis': 2091,
 'Business_&_Finance': 9878,
 'Other': 9306}

count = 0
for k,v in df_res.items():
    count += v

for k,v in df_res.items():
    print(k, v/count)

print('total:::', count)

In [ ]:
df_res = step2_save_filter_datas_gai(9, 9, 10, 10)
